# EAGLE3 Configuration Benchmark Analysis

Analyzes results from `scripts/bench_eagle3_configs.py`.

Experiments:
1. **Chain baseline**: topk=1, budget=steps (autoregressive drafting)
2. **Topk sweep**: How branching factor affects performance
3. **Steps sweep**: How tree depth affects performance
4. **Budget sweep**: How total draft tokens affects performance

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams.update({
    'font.size': 12,
    'figure.figsize': (12, 5),
    'figure.dpi': 100,
})

In [ ]:
# Load benchmark results
BENCH_PATH = "../results/qwen3_8b/eagle3_bench.json"  # adjust path as needed

with open(BENCH_PATH) as f:
    data = json.load(f)

vanilla_tpot = data["vanilla_tpot_ms"]
vanilla_tps = data["vanilla_tps"]
df = pd.DataFrame(data["results"])

print(f"Model: {data['model']}")
print(f"Draft: {data['draft_model']}")
print(f"Vanilla TPOT: {vanilla_tpot:.2f} ms/tok, TPS: {vanilla_tps:.1f}")
print(f"Configs: {len(df)}")
print()
df

## 1. Overview: All Configurations

In [ ]:
# Summary table sorted by speedup
cols = ['mode', 'topk', 'steps', 'budget', 'tpot_ms', 'speedup', 
        'mat', 'accept_rate', 'overall_tps', 'gpu_util_pct']
summary = df[cols].sort_values('speedup', ascending=False)
summary.style.background_gradient(subset=['speedup'], cmap='Greens') \
    .background_gradient(subset=['mat'], cmap='Blues') \
    .format({'speedup': '{:.2f}x', 'tpot_ms': '{:.1f}', 
             'mat': '{:.2f}', 'accept_rate': '{:.3f}',
             'overall_tps': '{:.1f}', 'gpu_util_pct': '{:.0f}%'})

In [ ]:
# Bar chart: speedup by config
fig, ax = plt.subplots(figsize=(14, 6))

labels = []
for _, r in df.iterrows():
    if r['mode'] == 'chain':
        labels.append(f"chain\nB={r['budget']}")
    else:
        labels.append(f"k={r['topk']}\ns={r['steps']}\nB={r['budget']}")

colors = ['#ff7f0e' if m == 'chain' else '#1f77b4' for m in df['mode']]
bars = ax.bar(range(len(df)), df['speedup'], color=colors)
ax.set_xticks(range(len(df)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Speedup vs Vanilla')
ax.set_title('EAGLE3 Speedup: Chain (orange) vs Tree (blue)')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Vanilla (1.0x)')

for bar, spd in zip(bars, df['speedup']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{spd:.2f}x', ha='center', va='bottom', fontsize=8)

ax.legend()
plt.tight_layout()
plt.show()

## 2. Chain Baseline: Effect of Chain Length

In [ ]:
chain = df[df['mode'] == 'chain'].sort_values('budget')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Speedup vs chain length
axes[0].plot(chain['budget'], chain['speedup'], 'o-', color='#ff7f0e', linewidth=2)
axes[0].set_xlabel('Chain Length (budget)')
axes[0].set_ylabel('Speedup')
axes[0].set_title('Speedup vs Chain Length')
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xscale('log', basex=2)

# MAT vs chain length
axes[1].plot(chain['budget'], chain['mat'], 's-', color='#2ca02c', linewidth=2)
axes[1].set_xlabel('Chain Length (budget)')
axes[1].set_ylabel('MAT (Mean Accepted Tokens)')
axes[1].set_title('MAT vs Chain Length')
axes[1].set_xscale('log', basex=2)

# TPOT breakdown
axes[2].plot(chain['budget'], chain['tpot_ms'], 'D-', color='#d62728', linewidth=2)
axes[2].set_xlabel('Chain Length (budget)')
axes[2].set_ylabel('TPOT (ms/tok)')
axes[2].set_title('Latency vs Chain Length')
axes[2].axhline(y=vanilla_tpot, color='gray', linestyle='--', alpha=0.5, label='Vanilla')
axes[2].legend()
axes[2].set_xscale('log', basex=2)

plt.tight_layout()
plt.show()

chain[['budget', 'tpot_ms', 'speedup', 'mat', 'accept_rate', 'overall_tps']]

## 3. Topk Sweep: Effect of Branching Factor

Fixed: steps=3, budget=32. Varying: topk=2,4,8 + chain(topk=1).

In [ ]:
# topk sweep: steps=3, budget=32 (trees) + chain budget=32 (topk=1)
topk_sweep = df[
    ((df['mode'] == 'tree') & (df['steps'] == 3) & (df['budget'] == 32)) |
    ((df['mode'] == 'chain') & (df['budget'] == 32))
].sort_values('topk')

if len(topk_sweep) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].plot(topk_sweep['topk'], topk_sweep['speedup'], 'o-', linewidth=2)
    axes[0].set_xlabel('topk (branching factor)')
    axes[0].set_ylabel('Speedup')
    axes[0].set_title('Speedup vs topk (steps=3, B=32)')
    
    axes[1].plot(topk_sweep['topk'], topk_sweep['mat'], 's-', color='#2ca02c', linewidth=2)
    axes[1].set_xlabel('topk')
    axes[1].set_ylabel('MAT')
    axes[1].set_title('MAT vs topk')
    
    axes[2].plot(topk_sweep['topk'], topk_sweep['tpot_ms'], 'D-', color='#d62728', linewidth=2)
    axes[2].set_xlabel('topk')
    axes[2].set_ylabel('TPOT (ms/tok)')
    axes[2].set_title('Latency vs topk')
    axes[2].axhline(y=vanilla_tpot, color='gray', linestyle='--', alpha=0.5, label='Vanilla')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    topk_sweep[['topk', 'steps', 'budget', 'tpot_ms', 'speedup', 'mat', 'accept_rate']]
else:
    print('No topk sweep data found')

## 4. Steps Sweep: Effect of Tree Depth

Fixed: topk=8, budget=32. Varying: steps=2,3,4,5.

In [ ]:
steps_sweep = df[
    (df['mode'] == 'tree') & (df['topk'] == 8) & (df['budget'] == 32)
].sort_values('steps')

if len(steps_sweep) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].plot(steps_sweep['steps'], steps_sweep['speedup'], 'o-', linewidth=2)
    axes[0].set_xlabel('steps (max tree depth)')
    axes[0].set_ylabel('Speedup')
    axes[0].set_title('Speedup vs steps (topk=8, B=32)')
    
    axes[1].plot(steps_sweep['steps'], steps_sweep['mat'], 's-', color='#2ca02c', linewidth=2)
    axes[1].set_xlabel('steps')
    axes[1].set_ylabel('MAT')
    axes[1].set_title('MAT vs steps')
    
    axes[2].plot(steps_sweep['steps'], steps_sweep['tpot_ms'], 'D-', color='#d62728', linewidth=2)
    axes[2].set_xlabel('steps')
    axes[2].set_ylabel('TPOT (ms/tok)')
    axes[2].set_title('Latency vs steps')
    axes[2].axhline(y=vanilla_tpot, color='gray', linestyle='--', alpha=0.5, label='Vanilla')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    steps_sweep[['topk', 'steps', 'budget', 'tpot_ms', 'speedup', 'mat', 'accept_rate']]
else:
    print('No steps sweep data found')

## 5. Budget Sweep: Effect of Total Draft Tokens

Fixed: topk=8, steps=3. Varying: budget=4,8,16,32,64.

In [ ]:
budget_sweep = df[
    (df['mode'] == 'tree') & (df['topk'] == 8) & (df['steps'] == 3)
].sort_values('budget')

# Add chain data for comparison
chain_for_compare = df[df['mode'] == 'chain'].sort_values('budget')

if len(budget_sweep) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Speedup
    axes[0].plot(budget_sweep['budget'], budget_sweep['speedup'], 'o-', 
                 linewidth=2, label='Tree (k=8,s=3)')
    axes[0].plot(chain_for_compare['budget'], chain_for_compare['speedup'], 's--',
                 color='#ff7f0e', linewidth=2, label='Chain (k=1)')
    axes[0].set_xlabel('Budget (num_draft_tokens)')
    axes[0].set_ylabel('Speedup')
    axes[0].set_title('Speedup: Tree vs Chain')
    axes[0].set_xscale('log', basex=2)
    axes[0].legend()
    
    # MAT
    axes[1].plot(budget_sweep['budget'], budget_sweep['mat'], 'o-',
                 linewidth=2, label='Tree')
    axes[1].plot(chain_for_compare['budget'], chain_for_compare['mat'], 's--',
                 color='#ff7f0e', linewidth=2, label='Chain')
    axes[1].set_xlabel('Budget')
    axes[1].set_ylabel('MAT')
    axes[1].set_title('MAT: Tree vs Chain')
    axes[1].set_xscale('log', basex=2)
    axes[1].legend()
    
    # TPOT
    axes[2].plot(budget_sweep['budget'], budget_sweep['tpot_ms'], 'o-',
                 linewidth=2, label='Tree')
    axes[2].plot(chain_for_compare['budget'], chain_for_compare['tpot_ms'], 's--',
                 color='#ff7f0e', linewidth=2, label='Chain')
    axes[2].axhline(y=vanilla_tpot, color='gray', linestyle='--', alpha=0.5, label='Vanilla')
    axes[2].set_xlabel('Budget')
    axes[2].set_ylabel('TPOT (ms/tok)')
    axes[2].set_title('Latency: Tree vs Chain')
    axes[2].set_xscale('log', basex=2)
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    pd.concat([budget_sweep, chain_for_compare]).sort_values(['budget', 'mode'])[[
        'mode', 'topk', 'steps', 'budget', 'tpot_ms', 'speedup', 'mat', 'accept_rate'
    ]]
else:
    print('No budget sweep data found')

## 6. Efficiency Analysis: Speedup vs Cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MAT vs Accept Rate (efficiency of draft tokens)
for mode in ['chain', 'tree']:
    subset = df[df['mode'] == mode]
    color = '#ff7f0e' if mode == 'chain' else '#1f77b4'
    axes[0].scatter(subset['accept_rate'], subset['mat'], 
                    c=color, s=subset['budget'] * 3, alpha=0.7, 
                    label=mode, edgecolors='black', linewidth=0.5)
    for _, r in subset.iterrows():
        axes[0].annotate(f"B={r['budget']}", (r['accept_rate'], r['mat']),
                        fontsize=7, ha='center', va='bottom')

axes[0].set_xlabel('Accept Rate (accepted / budget)')
axes[0].set_ylabel('MAT (Mean Accepted Tokens)')
axes[0].set_title('Draft Efficiency: MAT vs Accept Rate\n(size = budget)')
axes[0].legend()

# Speedup vs TPOT
for mode in ['chain', 'tree']:
    subset = df[df['mode'] == mode]
    color = '#ff7f0e' if mode == 'chain' else '#1f77b4'
    axes[1].scatter(subset['tpot_ms'], subset['speedup'],
                    c=color, s=subset['budget'] * 3, alpha=0.7,
                    label=mode, edgecolors='black', linewidth=0.5)
    for _, r in subset.iterrows():
        axes[1].annotate(f"B={r['budget']}", (r['tpot_ms'], r['speedup']),
                        fontsize=7, ha='center', va='bottom')

axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(x=vanilla_tpot, color='gray', linestyle='--', alpha=0.3, label='Vanilla TPOT')
axes[1].set_xlabel('TPOT (ms/tok)')
axes[1].set_ylabel('Speedup')
axes[1].set_title('Speedup vs Latency\n(size = budget)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. GPU Utilization Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GPU util vs budget
for mode in ['chain', 'tree']:
    subset = df[df['mode'] == mode].sort_values('budget')
    marker = 's' if mode == 'chain' else 'o'
    axes[0].plot(subset['budget'], subset['gpu_util_pct'], f'{marker}-',
                label=mode, linewidth=2)

axes[0].set_xlabel('Budget')
axes[0].set_ylabel('GPU Utilization (%)')
axes[0].set_title('GPU Utilization vs Budget')
axes[0].set_xscale('log', basex=2)
axes[0].legend()

# GPU memory vs budget
for mode in ['chain', 'tree']:
    subset = df[df['mode'] == mode].sort_values('budget')
    marker = 's' if mode == 'chain' else 'o'
    axes[1].plot(subset['budget'], subset['gpu_mem_mb'], f'{marker}-',
                label=mode, linewidth=2)

axes[1].set_xlabel('Budget')
axes[1].set_ylabel('GPU Memory (MB)')
axes[1].set_title('GPU Memory vs Budget')
axes[1].set_xscale('log', basex=2)
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Optimal Configuration

In [ ]:
if 'error' not in df.columns:
    df['error'] = None
valid = df[df['error'].isna()].copy()

best = valid.loc[valid['speedup'].idxmax()]
best_chain = valid[valid['mode'] == 'chain'].loc[valid[valid['mode'] == 'chain']['speedup'].idxmax()]
best_tree = valid[valid['mode'] == 'tree'].loc[valid[valid['mode'] == 'tree']['speedup'].idxmax()]

print("=" * 60)
print("OPTIMAL CONFIGURATIONS")
print("=" * 60)
print(f"\nBest overall: {best['mode']} topk={best['topk']}, steps={best['steps']}, "
      f"budget={best['budget']} → {best['speedup']:.2f}x")
print(f"  TPOT={best['tpot_ms']:.1f}ms, MAT={best['mat']:.2f}, "
      f"accept_rate={best['accept_rate']:.3f}, TPS={best['overall_tps']:.1f}")

print(f"\nBest chain:   topk={best_chain['topk']}, steps={best_chain['steps']}, "
      f"budget={best_chain['budget']} → {best_chain['speedup']:.2f}x")
print(f"  TPOT={best_chain['tpot_ms']:.1f}ms, MAT={best_chain['mat']:.2f}")

print(f"\nBest tree:    topk={best_tree['topk']}, steps={best_tree['steps']}, "
      f"budget={best_tree['budget']} → {best_tree['speedup']:.2f}x")
print(f"  TPOT={best_tree['tpot_ms']:.1f}ms, MAT={best_tree['mat']:.2f}")

print(f"\nTree advantage over chain: "
      f"{(best_tree['speedup'] / best_chain['speedup'] - 1) * 100:+.1f}%")
print("=" * 60)

## 9. Per-Category Analysis (from request-level data)

In [ ]:
# If request-level data is available in the JSON
# (not always saved depending on output format)
print("Per-category breakdown requires request-level data.")
print("Re-run bench with --n-requests 16 (2 per category) for more data.")

## 10. Raw Data Export

In [ ]:
# Export to CSV for external analysis
csv_path = BENCH_PATH.replace('.json', '.csv')
df.to_csv(csv_path, index=False)
print(f"Exported to {csv_path}")
print()
df.to_string()